# ⚠️ DO NOT RUN — OVERWRITES FROZEN V1

This notebook retrains and overwrites:
- models/baseline/lightgbm.joblib  (the frozen V1 model)
- models/preprocessing/tree_preprocessor.joblib
- data/modeling/train.csv, validation.csv

Running it changes the file hashes, and the dashboard will refuse
to start because they no longer match model_metadata.json.

Recovery: git checkout models/baseline/lightgbm.joblib



# 16b — Development validation split robustness redesign

This notebook redesigns only the train/validation allocation of complete development runs. Fixed test runs and test artifacts are protected by checksums and are never loaded for modeling or split selection.

### 1. Define protected inputs and outputs

**What this cell does:** Maps development inputs, existing split reports, candidate-report outputs, and protected test artifacts; then records checksums.

**Why it matters:** Validation redesign must not change or inspect the test set, and it must not accidentally reuse old 6,640-row probabilities.

**What to understand:** Test files are fingerprinted as opaque files only. Their features, labels, probabilities, and model performance are never loaded.

In [1]:
from pathlib import Path
from itertools import product
import gc, hashlib, json, shutil, time

import joblib
import lightgbm as lgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
assert ROOT.name == 'AdoptAI_V1'
REPORT_DIR = ROOT / 'reports'; FIGURE_DIR = REPORT_DIR / 'figures/validation_split_redesign'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_PATH = ROOT / 'data/interim/feature_dataset.csv'
TRAIN_PATH = ROOT / 'data/modeling/train.csv'; VALIDATION_PATH = ROOT / 'data/modeling/validation.csv'
TEST_PATH = ROOT / 'data/modeling/test.csv'; PREPROCESSED_DIR = ROOT / 'data/modeling/preprocessed'
RUN_REPORT_PATH = REPORT_DIR / 'final_split_by_run.csv'; MACHINE_REPORT_PATH = REPORT_DIR / 'final_split_by_machine.csv'
SUMMARY_REPORT_PATH = REPORT_DIR / 'final_split_summary.csv'
output_paths = {
    'all_runs': REPORT_DIR / 'all_runs_split_diagnostic.csv',
    'candidates': REPORT_DIR / 'validation_split_candidates.csv',
    'models': REPORT_DIR / 'validation_split_model_comparison.csv',
    'runs': REPORT_DIR / 'validation_split_run_metrics.csv',
    'machines': REPORT_DIR / 'validation_split_machine_metrics.csv',
    'recommendation': REPORT_DIR / 'validation_split_recommendation.csv',
}
required = [FEATURE_PATH, TRAIN_PATH, VALIDATION_PATH, TEST_PATH, RUN_REPORT_PATH, MACHINE_REPORT_PATH, SUMMARY_REPORT_PATH]
assert all(path.is_file() for path in required)
protected_test_paths = [TEST_PATH, PREPROCESSED_DIR/'X_test_tree.csv', PREPROCESSED_DIR/'X_test_linear.csv',
                        PREPROCESSED_DIR/'y_test.csv', PREPROCESSED_DIR/'test_identifiers.csv']
assert all(path.is_file() for path in protected_test_paths)
def sha256_file(path):
    digest=hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()
feature_hash_before=sha256_file(FEATURE_PATH)
test_hashes_before={str(path.relative_to(ROOT)):sha256_file(path) for path in protected_test_paths}
print(f'Feature dataset fingerprinted: {feature_hash_before}')
print(f'Protected test artifacts fingerprinted without loading: {len(test_hashes_before)}')
print('No old validation probabilities are inputs to this notebook.')

Feature dataset fingerprinted: 4f37c797627366e3210f31792855335edecace32a7ea6d3e38046b4013fb57fa
Protected test artifacts fingerprinted without loading: 5
No old validation probabilities are inputs to this notebook.


### 2. Build the all-run diagnostic and explain the current composition

**What this cell does:** Reads chronology and segment membership without the target from the pre-split feature dataset, merges the established run-level class counts, and summarizes the current splits.

**Why it matters:** Run composition can be diagnosed without model predictions, while fixed-test label files remain unopened.

**What to understand:** All-positive, all-negative, tiny, and dominant runs can destabilize aggregate validation metrics even when global scores look strong.

In [2]:
sequence = pd.read_csv(FEATURE_PATH, usecols=['machine_id','run_id','segment_id','timestamp'])
sequence['_timestamp_dt']=pd.to_datetime(sequence['timestamp'],errors='coerce',utc=True)
assert sequence['_timestamp_dt'].notna().all() and len(sequence)==133_016
current_by_run=pd.read_csv(RUN_REPORT_PATH); current_by_machine=pd.read_csv(MACHINE_REPORT_PATH)
current_summary=pd.read_csv(SUMMARY_REPORT_PATH)
chronology=(sequence.groupby(['machine_id','run_id']).agg(start_timestamp=('_timestamp_dt','min'),
             end_timestamp=('_timestamp_dt','max'),row_count=('run_id','size'),segment_count=('segment_id','nunique')).reset_index())
counts=current_by_run[['machine_id','run_id','positive_rows','negative_rows','positive_rate','split']].rename(
    columns={'positive_rows':'positive_count','negative_rows':'negative_count','split':'current_split'})
all_runs=chronology.merge(counts,on=['machine_id','run_id'],how='left',validate='one_to_one')
assert all_runs['current_split'].notna().all() and all_runs['row_count'].sum()==133_016
all_runs=all_runs.sort_values(['machine_id','start_timestamp','run_id']).reset_index(drop=True)
all_runs['chronological_position_within_machine']=all_runs.groupby('machine_id').cumcount()+1
all_runs['class_composition']=np.select([all_runs.positive_count.eq(0),all_runs.negative_count.eq(0)],
                                                ['all_negative','all_positive'],default='mixed')
all_runs['extremely_small_run']=all_runs.row_count.lt(500)
all_runs['unusual_positive_rate']=all_runs.positive_rate.lt(5)|all_runs.positive_rate.gt(95)
all_runs.to_csv(output_paths['all_runs'],index=False)
current_composition=(all_runs.groupby('current_split').agg(runs=('run_id','nunique'),rows=('row_count','sum'),
    positive_count=('positive_count','sum'),machines=('machine_id','nunique'),
    all_positive_runs=('class_composition',lambda x:int(x.eq('all_positive').sum())),
    all_negative_runs=('class_composition',lambda x:int(x.eq('all_negative').sum())),
    mixed_runs=('class_composition',lambda x:int(x.eq('mixed').sum()))).reset_index())
current_composition['positive_rate']=current_composition.positive_count/current_composition.rows
validation_runs=all_runs.loc[all_runs.current_split.eq('validation')].copy()
validation_runs['row_share']=validation_runs.row_count/validation_runs.row_count.sum()
validation_runs['positive_share']=validation_runs.positive_count/max(1,validation_runs.positive_count.sum())
print('Current split composition:'); display(current_composition)
print('Current validation run composition:'); display(validation_runs[['machine_id','run_id','row_count','positive_rate','class_composition','row_share','positive_share','extremely_small_run']])
print('Development chronology by machine:')
for machine_id,group in all_runs.groupby('machine_id'):
    print(f'\n{machine_id}')
    display(group[['chronological_position_within_machine','run_id','current_split','row_count','positive_rate','class_composition']])

Current split composition:


,current_split,runs,rows,positive_count,machines,all_positive_runs,all_negative_runs,mixed_runs,positive_rate
0,test,9,27944,11356,4,1,1,7,0.406384
1,train,39,82860,28198,4,8,8,23,0.340309
2,validation,7,22212,10913,4,3,1,3,0.491311


Current validation run composition:


,machine_id,run_id,row_count,positive_rate,class_composition,row_share,positive_share,extremely_small_run
6,0890dcc046c079acc4de4202,90c048bb-46c5-4345-a113-d3bc37b876cb,3024,0.0000,all_negative,0.136143,0.000000,False
18,7232bc533c21ce408d45d473,f3c83d3f-0563-4e35-aa47-2e79402b0105,1789,26.3835,mixed,0.080542,0.043251,False
19,7232bc533c21ce408d45d473,96f661cc-1233-4539-a814-c4c352953500,247,58.2996,mixed,0.011120,0.013195,True
20,7232bc533c21ce408d45d473,b1dd3935-56dd-4497-a77e-67128988e2c9,268,100.0000,all_positive,0.012066,0.024558,True
35,a0f8c86097e55fbfa506d057,0fc9db54-83d3-456a-b9ea-1aa8a6f66cf9,5,100.0000,all_positive,0.000225,0.000458,True
36,a0f8c86097e55fbfa506d057,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,8602,20.3092,mixed,0.387268,0.160084,False
53,d588df123ac0d0ce20b112ac,35472c29-8dd3-49be-8d6b-1984d158745d,8277,100.0000,all_positive,0.372636,0.758453,False


Development chronology by machine:

0890dcc046c079acc4de4202


...

,chronological_position_within_machine,run_id,current_split,row_count,positive_rate,class_composition
10,1,d88b15dd-1915-43ca-90ef-69f31ff4d9c1,train,1165,26.6953,mixed
11,2,ec61755d-b5be-42ac-875d-3123e92add7a,train,1302,4.7619,mixed
12,3,fac82c2e-545a-402a-80f8-3d1fccb72c68,train,1611,44.0720,mixed
13,4,19127a70-e60c-4b47-b3e6-71e89175c174,train,119,88.2353,mixed
14,5,f81f7ffe-2f6d-4479-937f-cc82e574c286,train,2206,56.8450,mixed
15,6,6c7905b0-5124-4d83-8709-a6176c981aec,train,518,70.8494,mixed
16,7,81d2476d-af90-434e-8518-502c7195564c,train,1870,40.0535,mixed
17,8,1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d,train,2190,38.3105,mixed
18,9,f3c83d3f-0563-4e35-aa47-2e79402b0105,validation,1789,26.3835,mixed
19,10,96f661cc-1233-4539-a814-c4c352953500,validation,247,58.2996,mixed



a0f8c86097e55fbfa506d057


,chronological_position_within_machine,run_id,current_split,row_count,positive_rate,class_composition
24,1,2e9f4457-2a7f-4647-87ed-b86ac6343d33,train,182,0.0000,all_negative
25,2,f24e9c1a-f11e-405c-b663-45fa6e25c405,train,3860,11.7358,mixed
26,3,373070d1-ba59-4244-88ab-2d44a21f4983,train,261,0.0000,all_negative
27,4,70577f8e-1430-4489-8602-2096521ab84e,train,5603,25.0223,mixed
28,5,fc583c4c-4107-450e-959d-c74895b57935,train,7440,70.9274,mixed
29,6,e4e219be-13ba-4874-aad6-93d0b6f5f289,train,3419,20.1813,mixed
30,7,67c192c0-13a4-427a-b8e3-193c8fd99883,train,6536,16.4933,mixed
31,8,28ca5ae9-faed-4da3-8350-e2605f94aa88,train,8125,45.1200,mixed
32,9,4c9f7e66-830f-4c54-bfd5-f4d898f5464e,train,3753,84.9987,mixed
33,10,ea7c3f9f-8a45-4892-8f21-a77488321622,train,533,100.0000,all_positive



d588df123ac0d0ce20b112ac


,chronological_position_within_machine,run_id,current_split,row_count,positive_rate,class_composition
39,1,af502e6e-1657-4248-8b91-ce01c14277b9,train,3087,7.2886,mixed
40,2,9753a7c0-658b-4111-8e40-d347a98a680a,train,377,0.0000,all_negative
41,3,d4a43abe-57cb-4513-b782-4b927879f358,train,774,0.0000,all_negative
42,4,22fc3390-8d4a-4b01-b11e-c10adffcab2d,train,620,0.0000,all_negative
43,5,1e0bd844-6541-4320-8474-1b01bab00799,train,160,0.0000,all_negative
44,6,e4d80512-c282-45fb-8d13-e4e33eebfd72,train,57,0.0000,all_negative
45,7,6a1c86bc-1e52-4235-ac04-6d94cf1c3c5e,train,697,42.8981,mixed
46,8,f630b56e-4353-401a-bf40-5ed332d157f9,train,50,100.0000,all_positive
47,9,00f4a034-46ce-498e-b844-f1aa52fc6800,train,90,100.0000,all_positive
48,10,cc2a1e74-06e7-4cb1-92cd-6b21a91f240f,train,87,100.0000,all_positive


### 3. Generate complete-run chronological candidate designs

**What this cell does:** Keeps fixed test run IDs unchanged and creates four alternative train/validation suffix designs plus the current reference.

**Why it matters:** Later complete runs provide realistic temporal validation, but row percentage alone cannot determine a robust composition.

**What to understand:** The composition-aware design considers prevalence, run domination, single-class runs, and validation size using development labels only; it never uses test behavior.

In [3]:
fixed_test_runs=set(all_runs.loc[all_runs.current_split.eq('test'),'run_id'])
development_runs=all_runs.loc[~all_runs.current_split.eq('test')].copy()
assert len(fixed_test_runs)==9 and development_runs.run_id.nunique()==46
assignments={'current_reference':dict(zip(development_runs.run_id,development_runs.current_split))}
def suffix_assignment(target_fraction):
    mapping={}
    for machine_id,group in development_runs.sort_values(['machine_id','start_timestamp']).groupby('machine_id'):
        group=group.reset_index(drop=True); total=group.row_count.sum(); choices=[]
        for val_run_count in range(2,len(group)):
            share=group.tail(val_run_count).row_count.sum()/total
            choices.append((abs(share-target_fraction),val_run_count))
        val_run_count=min(choices)[1]; val_ids=set(group.tail(val_run_count).run_id)
        mapping.update({run_id:('validation' if run_id in val_ids else 'train') for run_id in group.run_id})
    return mapping
for target in [0.20,0.25,0.30]: assignments[f'per_machine_suffix_{int(target*100)}pct']=suffix_assignment(target)
machine_groups={machine:g.reset_index(drop=True) for machine,g in development_runs.sort_values(['machine_id','start_timestamp']).groupby('machine_id')}
choice_ranges=[range(2,min(5,len(group)-1)+1) for group in machine_groups.values()]
best_objective=None; best_mapping=None
for counts_choice in product(*choice_ranges):
    mapping={}; val_parts=[]; train_parts=[]
    for (machine,group),val_count in zip(machine_groups.items(),counts_choice):
        val_ids=set(group.tail(val_count).run_id); val_parts.append(group[group.run_id.isin(val_ids)]); train_parts.append(group[~group.run_id.isin(val_ids)])
        mapping.update({run_id:('validation' if run_id in val_ids else 'train') for run_id in group.run_id})
    val=pd.concat(val_parts); train=pd.concat(train_parts); val_rows=val.row_count.sum(); val_pos=val.positive_count.sum()
    val_fraction=val_rows/development_runs.row_count.sum(); val_rate=val_pos/val_rows; train_rate=train.positive_count.sum()/train.row_count.sum()
    largest_run=val.row_count.max()/val_rows; largest_pos=val.positive_count.max()/max(1,val_pos)
    single_fraction=val.class_composition.ne('mixed').mean()
    objective=(1.8*abs(val_rate-train_rate)+abs(val_fraction-.25)+.45*largest_run+.35*largest_pos+.15*single_fraction)
    if best_objective is None or objective<best_objective: best_objective,best_mapping=objective,mapping
assignments['composition_aware_suffix']=best_mapping
DIFFICULT_RUN='baa2a6f4-7121-4bce-9c80-611ebc22ceaf'
candidate_rows=[]
for name,mapping in assignments.items():
    assert set(mapping)==set(development_runs.run_id) and set(mapping.values())=={'train','validation'}
    work=development_runs.assign(candidate_split=development_runs.run_id.map(mapping))
    train=work[work.candidate_split.eq('train')]; val=work[work.candidate_split.eq('validation')]
    chronology_valid=all(g.sort_values('start_timestamp').candidate_split.map({'train':0,'validation':1}).is_monotonic_increasing for _,g in work.groupby('machine_id'))
    test_after_validation=all((val.loc[val.machine_id.eq(m),'end_timestamp'].max()<all_runs.loc[(all_runs.machine_id.eq(m))&all_runs.current_split.eq('test'),'start_timestamp'].min()) for m in work.machine_id.unique())
    val_pos=val.positive_count.sum(); train_pos=train.positive_count.sum()
    candidate_rows.append({'candidate':name,'train_rows':int(train.row_count.sum()),'validation_rows':int(val.row_count.sum()),
      'validation_percentage_of_development':val.row_count.sum()/development_runs.row_count.sum(),
      'train_runs':len(train),'validation_runs':len(val),'validation_machines':val.machine_id.nunique(),
      'train_positive_rate':train_pos/train.row_count.sum(),'validation_positive_rate':val_pos/val.row_count.sum(),
      'absolute_prevalence_difference':abs(train_pos/train.row_count.sum()-val_pos/val.row_count.sum()),
      'validation_all_positive_runs':int(val.class_composition.eq('all_positive').sum()),
      'validation_all_negative_runs':int(val.class_composition.eq('all_negative').sum()),
      'validation_mixed_runs':int(val.class_composition.eq('mixed').sum()),
      'largest_validation_run_share':val.row_count.max()/val.row_count.sum(),
      'largest_validation_positive_share':val.positive_count.max()/max(1,val_pos),
      'chronology_valid':chronology_valid and test_after_validation,'run_overlap_count':0,
      'difficult_run_split':mapping[DIFFICULT_RUN],
      'train_run_ids':json.dumps(train.run_id.tolist()),'validation_run_ids':json.dumps(val.run_id.tolist()),
      'fixed_test_run_ids':json.dumps(sorted(fixed_test_runs))})
candidate_table=pd.DataFrame(candidate_rows)
assert candidate_table.chronology_valid.all() and candidate_table.run_overlap_count.eq(0).all()
display(candidate_table.drop(columns=['train_run_ids','validation_run_ids','fixed_test_run_ids']))

,candidate,train_rows,validation_rows,validation_percentage_of_development,train_runs,validation_runs,validation_machines,train_positive_rate,validation_positive_rate,absolute_prevalence_difference,validation_all_positive_runs,validation_all_negative_runs,validation_mixed_runs,largest_validation_run_share,largest_validation_positive_share,chronology_valid,run_overlap_count,difficult_run_split
0,current_reference,82860,22212,0.211398,39,7,4,0.340309,0.491311,0.151002,3,1,3,0.387268,0.758453,True,0,validation
1,per_machine_suffix_20pct,75607,29465,0.280427,36,10,4,0.344479,0.443441,0.098963,3,2,5,0.291940,0.633476,True,0,validation
2,per_machine_suffix_25pct,75074,29998,0.285499,35,11,4,0.339825,0.453330,0.113506,4,2,5,0.286752,0.608648,True,0,validation
3,per_machine_suffix_30pct,69131,35941,0.342061,33,13,4,0.310758,0.490470,0.179713,4,2,7,0.239337,0.469537,True,0,validation
4,composition_aware_suffix,70264,34808,0.331278,34,12,4,0.370318,0.376092,0.005774,3,2,7,0.247127,0.632266,True,0,validation


### 4. Load development rows and define train-only preprocessing and metrics

**What this cell does:** Loads only the current train and validation CSVs, reconstructs the 46-run development pool, and defines the existing leakage exclusions, train-only median imputation, LightGBM configuration, and subgroup metrics.

**Why it matters:** Every candidate must refit preprocessing and the model from its own training rows; reusing the active preprocessor would leak candidate-validation statistics.

**What to understand:** Single-class run PR-AUC and ROC-AUC are explicitly unavailable and excluded from run-level minimum/mean calculations.

In [4]:
development_data=pd.concat([pd.read_csv(TRAIN_PATH),pd.read_csv(VALIDATION_PATH)],ignore_index=True)
assert len(development_data)==105_072 and set(development_data.run_id)==set(development_runs.run_id)
assert not (set(development_data.run_id)&fixed_test_runs)
development_data['_timestamp_dt']=pd.to_datetime(development_data.timestamp,errors='coerce',utc=True)
development_data=development_data.sort_values(['machine_id','_timestamp_dt','run_id','segment_id']).reset_index(drop=True)
TARGET='slowdown_in_5min'; THRESHOLD=.50; RANDOM_SEED=42; N_JOBS=4
explicit_exclusions={'id','machine_id','run_id','segment_id','timestamp','slowdown_in_5min','slowdown_in_10min','valid_5min_horizon','valid_10min_horizon','slowdown_now','status','ended_at_utc','run_complete','phase','legacy_id','stress_cpu_target_pct','stress_memory_target_mb','missed_deadline','sensor_errors_json','sample_reliable','temperature_c','gpu_usage_pct','cpu_per_core_json','gpu_per_device_json'}
rule_c_prefixes=('moderate_','severe_','rule_a_','rule_b_','rule_c_')
def prohibited(column): return column in explicit_exclusions or column.startswith(rule_c_prefixes) or 'slowdown' in column.lower()
base_candidate_features=[c for c in development_data.columns if c!='_timestamp_dt' and not prohibited(c)]
def prepare_candidate(train_frame,val_frame,linear=False):
    audit=train_frame[base_candidate_features]
    retained=[c for c in base_candidate_features if pd.api.types.is_numeric_dtype(audit[c]) and not audit[c].isna().all() and audit[c].nunique(dropna=True)>1]
    pipeline=Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True))]+([('scaler',StandardScaler())] if linear else []))
    X_train=pipeline.fit_transform(train_frame[retained]); X_val=pipeline.transform(val_frame[retained])
    assert np.isfinite(X_train).all() and np.isfinite(X_val).all()
    return pipeline,retained,X_train,X_val
def binary_metrics(y_true,probability):
    y=np.asarray(y_true,dtype=int); p=np.asarray(probability); pred=(p>=THRESHOLD).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel(); mixed=np.unique(y).size==2
    return {'pr_auc':average_precision_score(y,p) if mixed else np.nan,'roc_auc':roc_auc_score(y,p) if mixed else np.nan,
      'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),'f1':f1_score(y,pred,zero_division=0),
      'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp),'mean_probability':float(p.mean()),'predicted_positive_rate':float(pred.mean())}
model_rows=[]; run_metric_rows=[]; machine_metric_rows=[]
def record_evaluation(candidate,variant,model,train_frame,val_frame,X_train,X_val,retained_count,transformed_count,best_iteration=np.nan):
    y_train=train_frame[TARGET].astype(int).to_numpy(); y_val=val_frame[TARGET].astype(int).to_numpy()
    train_prob=model.predict_proba(X_train)[:,1]; val_prob=model.predict_proba(X_val)[:,1]
    train_pr=average_precision_score(y_train,train_prob); global_result=binary_metrics(y_val,val_prob)
    context=val_frame[['machine_id','run_id','segment_id','timestamp',TARGET]].reset_index(drop=True).copy(); context['probability']=val_prob
    candidate_run_rows=[]
    for run_id,group in context.groupby('run_id',sort=False):
        result=binary_metrics(group[TARGET],group.probability); positives=int(group[TARGET].sum()); negatives=len(group)-positives
        row={'candidate':candidate,'model_variant':variant,'machine_id':group.machine_id.iloc[0],'run_id':run_id,'rows':len(group),
             'positive_count':positives,'negative_count':negatives,'positive_rate':positives/len(group),
             'class_composition':'all_negative' if positives==0 else 'all_positive' if negatives==0 else 'mixed',
             'informative_mixed_run':bool(positives>0 and negatives>0 and len(group)>=500),**result}
        candidate_run_rows.append(row); run_metric_rows.append(row)
    for machine_id,group in context.groupby('machine_id',sort=False):
        result=binary_metrics(group[TARGET],group.probability); positives=int(group[TARGET].sum()); negatives=len(group)-positives
        machine_metric_rows.append({'candidate':candidate,'model_variant':variant,'machine_id':machine_id,'rows':len(group),
          'positive_count':positives,'negative_count':negatives,'positive_rate':positives/len(group),
          'class_composition':'all_negative' if positives==0 else 'all_positive' if negatives==0 else 'mixed',**result})
    run_df=pd.DataFrame(candidate_run_rows); informative=run_df[run_df.informative_mixed_run]
    model_rows.append({'candidate':candidate,'model_variant':variant,'train_rows':len(train_frame),'validation_rows':len(val_frame),
      'retained_original_features':retained_count,'transformed_features':transformed_count,'train_pr_auc':train_pr,
      'validation_pr_auc':global_result['pr_auc'],'train_validation_pr_auc_gap':train_pr-global_result['pr_auc'],
      **{k:global_result[k] for k in ['roc_auc','precision','recall','f1','tn','fp','fn','tp']},
      'minimum_informative_run_pr_auc':informative.pr_auc.min(),'minimum_informative_run_recall':informative.recall.min(),
      'mean_informative_run_pr_auc':informative.pr_auc.mean(),'std_informative_run_pr_auc':informative.pr_auc.std(ddof=0),
      'mean_informative_run_recall':informative.recall.mean(),'informative_run_count':len(informative),
      'largest_validation_run_share':run_df.rows.max()/len(val_frame),'best_iteration':best_iteration})
    return global_result

/var/folders/3h/tpfh764d3msgh0j4_bys88yr0000gn/T/ipykernel_39584/1768949690.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  development_data['_timestamp_dt']=pd.to_datetime(development_data.timestamp,errors='coerce',utc=True)


### 5. Fit the baseline LightGBM on every candidate split

**What this cell does:** Refits train-only preprocessing and the unchanged 300-tree baseline LightGBM for every candidate, then records global, machine, and run metrics.

**Why it matters:** Candidate validation sets must be compared using models trained under identical rules, not using predictions from a different split.

**What to understand:** Threshold 0.50 supplies diagnostic FP/FN counts; ranking robustness and informative-run stability remain the primary evidence.

In [5]:
for candidate,mapping in assignments.items():
    train_ids=[r for r,s in mapping.items() if s=='train']; val_ids=[r for r,s in mapping.items() if s=='validation']
    train_frame=development_data[development_data.run_id.isin(train_ids)].copy(); val_frame=development_data[development_data.run_id.isin(val_ids)].copy()
    preprocessor,retained,X_train,X_val=prepare_candidate(train_frame,val_frame)
    ratio=float((train_frame[TARGET]==0).sum()/(train_frame[TARGET]==1).sum())
    model=LGBMClassifier(objective='binary',n_estimators=300,learning_rate=.05,num_leaves=31,scale_pos_weight=ratio,random_state=RANDOM_SEED,n_jobs=N_JOBS,verbosity=-1)
    started=time.perf_counter(); model.fit(X_train,train_frame[TARGET].astype(int))
    result=record_evaluation(candidate,'baseline',model,train_frame,val_frame,X_train,X_val,len(retained),X_train.shape[1])
    print(f"{candidate}: rows={len(val_frame):,}, PR-AUC={result['pr_auc']:.6f}, recall={result['recall']:.4f}, seconds={time.perf_counter()-started:.2f}")
    del train_frame,val_frame,preprocessor,X_train,X_val,model; gc.collect()
baseline_comparison=pd.DataFrame(model_rows)
candidate_table=candidate_table.merge(baseline_comparison[['candidate','validation_pr_auc','minimum_informative_run_pr_auc','minimum_informative_run_recall','std_informative_run_pr_auc']],on='candidate',how='left')
candidate_table.to_csv(output_paths['candidates'],index=False)
display(baseline_comparison.sort_values('validation_pr_auc',ascending=False))

/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


current_reference: rows=22,212, PR-AUC=0.968988, recall=0.8997, seconds=4.64


/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

per_machine_suffix_30pct: rows=35,941, PR-AUC=0.961889, recall=0.9218, seconds=4.22


/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


composition_aware_suffix: rows=34,808, PR-AUC=0.662668, recall=0.8814, seconds=4.30


,candidate,model_variant,train_rows,validation_rows,retained_original_features,transformed_features,train_pr_auc,validation_pr_auc,train_validation_pr_auc_gap,roc_auc,...,fn,tp,minimum_informative_run_pr_auc,minimum_informative_run_recall,mean_informative_run_pr_auc,std_informative_run_pr_auc,mean_informative_run_recall,informative_run_count,largest_validation_run_share,best_iteration
0,current_reference,baseline,82860,22212,228,360,0.999996,0.968988,0.031008,0.958868,...,1095,9818,0.572310,0.457928,0.661295,0.088985,0.628328,2,0.387268,NaN
1,per_machine_suffix_20pct,baseline,75607,29465,228,360,0.999996,0.962049,0.037947,0.951986,...,1171,11895,0.034649,0.321429,0.600032,0.358722,0.656746,4,0.291940,NaN
3,per_machine_suffix_30pct,baseline,69131,35941,228,360,0.999998,0.961889,0.038109,0.945657,...,1378,16250,0.057002,0.429880,0.683083,0.320735,0.820561,6,0.239337,NaN
2,per_machine_suffix_25pct,baseline,75074,29998,228,360,0.999996,0.961242,0.038754,0.948699,...,1235,12364,0.038648,0.369048,0.586475,0.356523,0.656329,4,0.286752,NaN
4,composition_aware_suffix,baseline,70264,34808,228,360,1.000000,0.662668,0.337331,0.786196,...,1553,11538,0.055743,0.445907,0.564871,0.285814,0.778378,6,0.247127,NaN


### 6. Compare early stopping on the strongest designs

**What this cell does:** Ranks baseline designs with a stability-aware score, then evaluates early stopping on the two strongest alternatives and the current reference.

**Why it matters:** Early stopping may reduce the near-perfect training fit, but it must also preserve difficult-run behavior and cannot be judged by global PR-AUC alone.

**What to understand:** Validation is used only for early-stopping monitoring; no broad hyperparameter search or test evaluation occurs.

In [6]:
composition_lookup=candidate_table.set_index('candidate')
baseline_comparison['design_score']=(.28*baseline_comparison.validation_pr_auc+.27*baseline_comparison.minimum_informative_run_pr_auc+
 .18*baseline_comparison.minimum_informative_run_recall+.12*baseline_comparison.mean_informative_run_pr_auc-
 .08*baseline_comparison.std_informative_run_pr_auc-.04*baseline_comparison.candidate.map(composition_lookup.absolute_prevalence_difference)-
 .03*baseline_comparison.candidate.map(composition_lookup.largest_validation_run_share))
alternative_top=(baseline_comparison[~baseline_comparison.candidate.eq('current_reference')].sort_values('design_score',ascending=False).head(2).candidate.tolist())
early_candidates=['current_reference',*alternative_top]
for candidate in early_candidates:
    mapping=assignments[candidate]; train_ids=[r for r,s in mapping.items() if s=='train']; val_ids=[r for r,s in mapping.items() if s=='validation']
    train_frame=development_data[development_data.run_id.isin(train_ids)].copy(); val_frame=development_data[development_data.run_id.isin(val_ids)].copy()
    preprocessor,retained,X_train,X_val=prepare_candidate(train_frame,val_frame)
    ratio=float((train_frame[TARGET]==0).sum()/(train_frame[TARGET]==1).sum())
    model=LGBMClassifier(objective='binary',n_estimators=1000,learning_rate=.05,num_leaves=31,max_depth=-1,min_child_samples=20,
                         scale_pos_weight=ratio,random_state=RANDOM_SEED,n_jobs=N_JOBS,verbosity=-1)
    model.fit(X_train,train_frame[TARGET].astype(int),eval_set=[(X_val,val_frame[TARGET].astype(int))],eval_metric='average_precision',
              callbacks=[lgb.early_stopping(50,first_metric_only=True,verbose=False),lgb.log_evaluation(0)])
    result=record_evaluation(candidate,'early_stopping',model,train_frame,val_frame,X_train,X_val,len(retained),X_train.shape[1],int(model.best_iteration_))
    print(f"{candidate} early stopping: best_iteration={model.best_iteration_}, PR-AUC={result['pr_auc']:.6f}")
    del train_frame,val_frame,preprocessor,X_train,X_val,model; gc.collect()
model_comparison=pd.DataFrame(model_rows)
run_metrics=pd.DataFrame(run_metric_rows); machine_metrics=pd.DataFrame(machine_metric_rows)
model_comparison.to_csv(output_paths['models'],index=False); run_metrics.to_csv(output_paths['runs'],index=False); machine_metrics.to_csv(output_paths['machines'],index=False)
display(model_comparison.sort_values(['candidate','model_variant']))

/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


current_reference early stopping: best_iteration=60, PR-AUC=0.970804


/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


per_machine_suffix_30pct early stopping: best_iteration=73, PR-AUC=0.967357


per_machine_suffix_25pct early stopping: best_iteration=30, PR-AUC=0.965039


/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,candidate,model_variant,train_rows,validation_rows,retained_original_features,transformed_features,train_pr_auc,validation_pr_auc,train_validation_pr_auc_gap,roc_auc,...,fn,tp,minimum_informative_run_pr_auc,minimum_informative_run_recall,mean_informative_run_pr_auc,std_informative_run_pr_auc,mean_informative_run_recall,informative_run_count,largest_validation_run_share,best_iteration
4,composition_aware_suffix,baseline,70264,34808,228,360,1.000000,0.662668,0.337331,0.786196,...,1553,11538,0.055743,0.445907,0.564871,0.285814,0.778378,6,0.247127,NaN
0,current_reference,baseline,82860,22212,228,360,0.999996,0.968988,0.031008,0.958868,...,1095,9818,0.572310,0.457928,0.661295,0.088985,0.628328,2,0.387268,NaN
5,current_reference,early_stopping,82860,22212,228,360,0.992359,0.970804,0.021555,0.963428,...,1037,9876,0.546426,0.506583,0.658039,0.111613,0.626173,2,0.387268,60.0
1,per_machine_suffix_20pct,baseline,75607,29465,228,360,0.999996,0.962049,0.037947,0.951986,...,1171,11895,0.034649,0.321429,0.600032,0.358722,0.656746,4,0.291940,NaN
2,per_machine_suffix_25pct,baseline,75074,29998,228,360,0.999996,0.961242,0.038754,0.948699,...,1235,12364,0.038648,0.369048,0.586475,0.356523,0.656329,4,0.286752,NaN
7,per_machine_suffix_25pct,early_stopping,75074,29998,228,360,0.979068,0.965039,0.014030,0.960108,...,1259,12340,0.032045,0.083333,0.598420,0.361284,0.588859,4,0.286752,30.0
3,per_machine_suffix_30pct,baseline,69131,35941,228,360,0.999998,0.961889,0.038109,0.945657,...,1378,16250,0.057002,0.429880,0.683083,0.320735,0.820561,6,0.239337,NaN
6,per_machine_suffix_30pct,early_stopping,69131,35941,228,360,0.994446,0.967357,0.027088,0.955649,...,1278,16350,0.027201,0.297619,0.687185,0.331948,0.736468,6,0.239337,73.0


### 7. Recommend a robust development design and decide whether replacement is justified

**What this cell does:** Scores candidate/model combinations using chronology, coverage, global ranking, difficult-run stability, variation, prevalence, and domination; then compares the winner with the current reference.

**Why it matters:** The highest global PR-AUC is not automatically the safest validation strategy. An active split changes only when improvement is broad and material.

**What to understand:** The difficult run remains represented according to chronology. A strict clear-improvement gate prevents unnecessary active-artifact churn.

In [7]:
comparison=model_comparison.merge(candidate_table[['candidate','validation_runs','validation_machines','train_positive_rate','validation_positive_rate','absolute_prevalence_difference','validation_all_positive_runs','validation_all_negative_runs','largest_validation_positive_share','chronology_valid','difficult_run_split']],on='candidate',how='left')
comparison['robustness_score']=(.24*comparison.validation_pr_auc+.27*comparison.minimum_informative_run_pr_auc+.20*comparison.minimum_informative_run_recall+
 .10*comparison.mean_informative_run_pr_auc-.08*comparison.std_informative_run_pr_auc-.04*comparison.absolute_prevalence_difference-
 .035*comparison.largest_validation_run_share-.035*comparison.largest_validation_positive_share)
eligible=comparison[comparison.chronology_valid & comparison.validation_machines.eq(4) & comparison.validation_runs.ge(8)].copy()
recommended=eligible.sort_values(['robustness_score','minimum_informative_run_pr_auc','minimum_informative_run_recall'],ascending=False).iloc[0]
current=comparison[(comparison.candidate.eq('current_reference'))&(comparison.model_variant.eq('baseline'))].iloc[0]
clearly_better=bool(recommended.candidate!='current_reference' and recommended.validation_pr_auc>=current.validation_pr_auc-.02 and
    recommended.minimum_informative_run_pr_auc>=current.minimum_informative_run_pr_auc+.03 and
    recommended.minimum_informative_run_recall>=current.minimum_informative_run_recall+.03 and
    recommended.std_informative_run_pr_auc<=current.std_informative_run_pr_auc+.01 and
    recommended.largest_validation_run_share<current.largest_validation_run_share)
difficult_history=all_runs.loc[all_runs.run_id.eq(DIFFICULT_RUN)].iloc[0]
recommendation=pd.DataFrame([{'recommended_candidate':recommended.candidate,'recommended_model_variant':recommended.model_variant,
 'robustness_score':recommended.robustness_score,'clear_improvement_over_current':clearly_better,
 'active_split_update_required':clearly_better,'test_runs_fixed':True,'test_artifacts_loaded':False,
 'difficult_run_id':DIFFICULT_RUN,'difficult_run_machine':difficult_history.machine_id,
 'difficult_run_chronological_position':difficult_history.chronological_position_within_machine,
 'difficult_run_positive_rate':difficult_history.positive_rate,'difficult_run_recommended_split':recommended.difficult_run_split,
 'threshold_optimization_recommendation':'proceed_only_after_human_review' if clearly_better else 'do_not_proceed'}])
recommendation.to_csv(output_paths['recommendation'],index=False)
print('Recommended development design/model:'); display(recommended.to_frame('value'))
print(f'Clear improvement sufficient to replace active split: {clearly_better}')
print('Difficult-run placement and candidate metrics:')
display(candidate_table[['candidate','difficult_run_split']].merge(run_metrics[run_metrics.run_id.eq(DIFFICULT_RUN)],on='candidate',how='left'))

Recommended development design/model:


,value
candidate,per_machine_suffix_30pct
model_variant,baseline
train_rows,69131
validation_rows,35941
retained_original_features,228
transformed_features,360
train_pr_auc,0.999998
validation_pr_auc,0.961889
train_validation_pr_auc_gap,0.038109
roc_auc,0.945657


Clear improvement sufficient to replace active split: False
Difficult-run placement and candidate metrics:


,candidate,difficult_run_split,model_variant,machine_id,run_id,rows,positive_count,negative_count,positive_rate,class_composition,...,roc_auc,precision,recall,f1,tn,fp,fn,tp,mean_probability,predicted_positive_rate
0,current_reference,validation,baseline,a0f8c86097e55fbfa506d057,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,8602,1747,6855,0.203092,mixed,...,0.745536,0.562588,0.457928,0.504891,6233,622,947,800,0.269570,0.165310
1,current_reference,validation,early_stopping,a0f8c86097e55fbfa506d057,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,8602,1747,6855,0.203092,mixed,...,0.779179,0.403189,0.506583,0.449011,5545,1310,862,885,0.392231,0.255173
2,per_machine_suffix_20pct,validation,baseline,a0f8c86097e55fbfa506d057,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,8602,1747,6855,0.203092,mixed,...,0.758435,0.550308,0.460218,0.501247,6198,657,943,804,0.278919,0.169844
3,per_machine_suffix_25pct,validation,baseline,a0f8c86097e55fbfa506d057,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,8602,1747,6855,0.203092,mixed,...,0.703536,0.526734,0.417287,0.465666,6200,655,1018,729,0.280380,0.160893
4,per_machine_suffix_25pct,validation,early_stopping,a0f8c86097e55fbfa506d057,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,8602,1747,6855,0.203092,mixed,...,0.810316,0.478987,0.508872,0.493478,5888,967,858,889,0.377769,0.215764
5,per_machine_suffix_30pct,validation,baseline,a0f8c86097e55fbfa506d057,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,8602,1747,6855,0.203092,mixed,...,0.717453,0.564237,0.429880,0.487979,6275,580,996,751,0.244319,0.154731
6,per_machine_suffix_30pct,validation,early_stopping,a0f8c86097e55fbfa506d057,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,8602,1747,6855,0.203092,mixed,...,0.770083,0.483561,0.547224,0.513426,5834,1021,791,956,0.376777,0.229830
7,composition_aware_suffix,validation,baseline,a0f8c86097e55fbfa506d057,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,8602,1747,6855,0.203092,mixed,...,0.688193,0.421537,0.445907,0.433380,5786,1069,968,779,0.308909,0.214834


### 8. Create diagnostic figures

**What this cell does:** Visualizes run chronology and positive rates, candidate prevalence, global ranking, minimum run performance, recall stability, and validation domination.

**Why it matters:** Visual diagnostics expose whether a recommendation is driven by broad coverage or one unusually large or single-class run.

**What to understand:** These plots support development-split review; they are not threshold or test-performance figures.

In [8]:
figure_paths=[]
fig,axes=plt.subplots(2,2,figsize=(15,10),sharey=True)
colors={'train':'tab:blue','validation':'tab:orange','test':'tab:red'}
for ax,(machine,g) in zip(axes.ravel(),all_runs.groupby('machine_id')):
    for split,part in g.groupby('current_split'): ax.scatter(part.start_timestamp,part.positive_rate,s=np.clip(part.row_count/20,20,250),label=split,color=colors[split],alpha=.75)
    ax.set_title(machine); ax.set_ylabel('Positive rate (%)'); ax.tick_params(axis='x',rotation=25); ax.grid(alpha=.2); ax.legend()
fig.suptitle('Positive rate by complete run over time'); fig.tight_layout(); p=FIGURE_DIR/'positive_rate_by_run_over_time.png'; fig.savefig(p,dpi=150); plt.close(fig); figure_paths.append(p)
plot_specs=[('validation_positive_rate','Candidate validation prevalence','Positive rate','candidate_validation_prevalence.png'),
 ('validation_pr_auc','Candidate baseline global PR-AUC','PR-AUC','candidate_global_pr_auc.png'),
 ('minimum_informative_run_pr_auc','Minimum informative-run PR-AUC','PR-AUC','candidate_minimum_run_pr_auc.png'),
 ('minimum_informative_run_recall','Minimum informative-run recall','Recall','candidate_minimum_run_recall.png'),
 ('std_informative_run_pr_auc','Informative-run PR-AUC variability','Standard deviation','candidate_run_variability.png'),
 ('largest_validation_run_share','Largest run contribution to validation','Row share','candidate_largest_run_share.png')]
plot_data=comparison.sort_values(['candidate','model_variant'])
for column,title,ylabel,filename in plot_specs:
    fig,ax=plt.subplots(figsize=(11,5)); labels=plot_data.candidate+' / '+plot_data.model_variant; ax.bar(labels,plot_data[column]); ax.set(title=title,ylabel=ylabel); ax.tick_params(axis='x',rotation=30); ax.grid(axis='y',alpha=.25); fig.tight_layout(); p=FIGURE_DIR/filename; fig.savefig(p,dpi=150); plt.close(fig); figure_paths.append(p)
print(f'Saved {len(figure_paths)} figures under {FIGURE_DIR}')

Saved 7 figures under /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/figures/validation_split_redesign


### 9. Apply the strict replacement gate and verify protected artifacts

**What this cell does:** Archives the current 153k development artifacts and updates active train/validation outputs only if the strict robustness gate passed; otherwise it leaves them unchanged. It always verifies all protected test checksums.

**Why it matters:** A plausible alternative is not enough to justify replacing reproducible active artifacts, and test protection must remain provable.

**What to understand:** When an update occurs, test files remain byte-for-byte unchanged and are not transformed with the new provisional preprocessor until a model strategy is frozen.

In [9]:
active_files_modified=[]; archive_dir=ROOT/'artifacts/archive/153521_pre_validation_redesign_20260814'
if clearly_better:
    archive_targets=[TRAIN_PATH,VALIDATION_PATH,SUMMARY_REPORT_PATH,RUN_REPORT_PATH,MACHINE_REPORT_PATH,
      ROOT/'models/preprocessing/tree_preprocessor.joblib',ROOT/'models/preprocessing/linear_preprocessor.joblib',ROOT/'models/baseline/lightgbm.joblib',
      REPORT_DIR/'preprocessing_summary.csv',REPORT_DIR/'preprocessing_feature_report.csv',REPORT_DIR/'baseline_model_comparison.csv',
      REPORT_DIR/'robustness_by_machine.csv',REPORT_DIR/'robustness_by_run.csv',REPORT_DIR/'overfitting_robustness_summary.csv']
    for source in archive_targets:
        if source.exists():
            destination=archive_dir/source.relative_to(ROOT); destination.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(source,destination)
    mapping=assignments[recommended.candidate]; train_ids=[r for r,s in mapping.items() if s=='train']; val_ids=[r for r,s in mapping.items() if s=='validation']
    active_train=development_data[development_data.run_id.isin(train_ids)].drop(columns='_timestamp_dt').copy()
    active_validation=development_data[development_data.run_id.isin(val_ids)].drop(columns='_timestamp_dt').copy()
    sort_cols=['machine_id','timestamp','run_id','segment_id']; active_train=active_train.sort_values(sort_cols); active_validation=active_validation.sort_values(sort_cols)
    active_train.to_csv(TRAIN_PATH,index=False); active_validation.to_csv(VALIDATION_PATH,index=False); active_files_modified += [TRAIN_PATH,VALIDATION_PATH]
    tree_preprocessor,retained,X_train_tree,X_validation_tree=prepare_candidate(active_train,active_validation)
    linear_preprocessor,linear_retained,X_train_linear,X_validation_linear=prepare_candidate(active_train,active_validation,linear=True); assert retained==linear_retained
    indicator_indices=tree_preprocessor.named_steps['imputer'].indicator_.features_.astype(int).tolist(); transformed_names=retained+[f'{retained[i]}__missing_indicator' for i in indicator_indices]
    for name,array in [('X_train_tree.csv',X_train_tree),('X_validation_tree.csv',X_validation_tree),('X_train_linear.csv',X_train_linear),('X_validation_linear.csv',X_validation_linear)]:
        pd.DataFrame(array,columns=transformed_names).to_csv(PREPROCESSED_DIR/name,index=False); active_files_modified.append(PREPROCESSED_DIR/name)
    active_train[[TARGET]].astype('int8').to_csv(PREPROCESSED_DIR/'y_train.csv',index=False); active_validation[[TARGET]].astype('int8').to_csv(PREPROCESSED_DIR/'y_validation.csv',index=False)
    identifier_columns=[c for c in ['id','machine_id','run_id','segment_id','timestamp','valid_5min_horizon','valid_10min_horizon','slowdown_in_10min'] if c in active_train]
    active_train[identifier_columns].to_csv(PREPROCESSED_DIR/'train_identifiers.csv',index=False); active_validation[identifier_columns].to_csv(PREPROCESSED_DIR/'validation_identifiers.csv',index=False)
    model_dir=ROOT/'models/preprocessing'; model_dir.mkdir(parents=True,exist_ok=True); joblib.dump(tree_preprocessor,model_dir/'tree_preprocessor.joblib'); joblib.dump(linear_preprocessor,model_dir/'linear_preprocessor.joblib')
    ratio=float((active_train[TARGET]==0).sum()/(active_train[TARGET]==1).sum()); baseline=LGBMClassifier(objective='binary',n_estimators=300,learning_rate=.05,num_leaves=31,scale_pos_weight=ratio,random_state=42,n_jobs=N_JOBS,verbosity=-1); baseline.fit(X_train_tree,active_train[TARGET].astype(int)); joblib.dump(baseline,ROOT/'models/baseline/lightgbm.joblib')
    early=LGBMClassifier(objective='binary',n_estimators=1000,learning_rate=.05,num_leaves=31,max_depth=-1,min_child_samples=20,scale_pos_weight=ratio,random_state=42,n_jobs=N_JOBS,verbosity=-1); early.fit(X_train_tree,active_train[TARGET].astype(int),eval_set=[(X_validation_tree,active_validation[TARGET].astype(int))],eval_metric='average_precision',callbacks=[lgb.early_stopping(50,first_metric_only=True,verbose=False),lgb.log_evaluation(0)]); joblib.dump(early,ROOT/'models/diagnostic/early_stopping_lightgbm.joblib')
    notice=pd.DataFrame([{'artifact':'protected test CSV and preprocessed test matrices','status':'unchanged_not_loaded','warning':'Existing test transforms belong to the pre-redesign preprocessor and must not be used until the development model is frozen and test is transformed once.'}]); notice.to_csv(REPORT_DIR/'test_protection_notice.csv',index=False)
    recommendation.loc[0,'archive_path']=str(archive_dir); recommendation.loc[0,'active_split_updated']=True
else:
    recommendation.loc[0,'archive_path']='not_created_no_active_replacement'; recommendation.loc[0,'active_split_updated']=False
recommendation.to_csv(output_paths['recommendation'],index=False)
assert sha256_file(FEATURE_PATH)==feature_hash_before
test_hashes_after={str(path.relative_to(ROOT)):sha256_file(path) for path in protected_test_paths}
assert test_hashes_after==test_hashes_before
assert all(path.exists() and path.stat().st_size>0 for path in [*output_paths.values(),*figure_paths])
assert len(pd.read_csv(output_paths['all_runs']))==55 and len(pd.read_csv(output_paths['candidates']))==5
print('FINAL VALIDATION-REDESIGN REPORT')
print(f'Recommended candidate: {recommended.candidate}; model={recommended.model_variant}')
print(f"Train rows={int(recommended.train_rows):,}; validation rows={int(recommended.validation_rows):,}; validation runs={int(recommended.validation_runs)}; machines={int(recommended.validation_machines)}")
print(f"Train positive rate={recommended.train_positive_rate:.2%}; validation positive rate={recommended.validation_positive_rate:.2%}")
print(f"Global PR-AUC={recommended.validation_pr_auc:.6f}; minimum informative-run PR-AUC={recommended.minimum_informative_run_pr_auc:.6f}; minimum recall={recommended.minimum_informative_run_recall:.6f}; run PR-AUC std={recommended.std_informative_run_pr_auc:.6f}")
print(f'Active split updated: {bool(recommendation.loc[0,"active_split_updated"])}')
print(f'Test artifacts unchanged byte-for-byte: {test_hashes_after==test_hashes_before}')
print('STOP: no threshold optimization, calibration, test evaluation, SHAP, or dashboard work was performed.')

FINAL VALIDATION-REDESIGN REPORT
Recommended candidate: per_machine_suffix_30pct; model=baseline
Train rows=69,131; validation rows=35,941; validation runs=13; machines=4
Train positive rate=31.08%; validation positive rate=49.05%
Global PR-AUC=0.961889; minimum informative-run PR-AUC=0.057002; minimum recall=0.429880; run PR-AUC std=0.320735
Active split updated: False
Test artifacts unchanged byte-for-byte: True
STOP: no threshold optimization, calibration, test evaluation, SHAP, or dashboard work was performed.
